# 20장 실습. 계층형 멀티 에이전트 위임

멀티 에이전트 시스템에는 다음 일을 누구에게 맡길지 정하는 감독(supervisor)이 있다. 흔히 LLM 이 그 자리에 앉는데, 홉마다 문장을 생성하므로 느리고 답을 파싱해야 한다. 감독의 일은 결국 선택지 몇 개 중 하나를 고르고, 일이 끝났는지 판단하는 것이다. 결정 모델의 일이다.

```
작업 ─▶ supervisor (next_agent: choice, done: noul)
           ├─ done ≥ 0.8 또는 홉 초과 ─▶ 끝
           └─ researcher · coder · writer · analyst ─▶ 결과를 기록에 붙이고 supervisor 로
```

| 절 | 내용 |
|---|---|
| 20.1 | 위임 질문 — 맡을 에이전트와 우선순위 (Jev) |
| 20.2 | LangGraph 감독 그래프 — 여러 단계 작업을 끝까지 |
| 20.3 | 루프 종료 — done 판단이 없으면 |
| 20.4 | 용도별 증류 — 로컬 LLM 이 만든 작업 문장과 실행 기록 |
| 20.5 | 엔진 비교 — Jev · Laya 기본 · Laya 증류 · 로컬 LLM |

평가 데이터는 저자가 쓴 단일 작업 40개(`tasks_ko.jsonl`, 정답 에이전트·우선순위)와 여러 단계 작업 8개(`scenarios_ko.jsonl`)다. 시나리오의 에이전트는 실제 일을 하지 않는다. 맞는 차례에 불리면 정해 둔 결과를 내고, 아니면 「할 일이 없다」고 답한다. 그래서 감독의 판단만 따로 잴 수 있다.

In [1]:
import os, json, time, random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, matplotlib.pyplot as plt, requests
import koreanize_matplotlib  # noqa: F401
from dotenv import load_dotenv
from requests.adapters import HTTPAdapter, Retry

load_dotenv()
DATA, OUT = Path("data"), Path("out")
OUT.mkdir(exist_ok=True)
TASKS = ["topic", "entails", "positive", "similarity"]
COLOR = {"topic": "#2a78d6", "entails": "#eb6834", "positive": "#1baf7a", "similarity": "#eda100"}
pd.set_option("display.precision", 3)
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False})
read_jsonl = lambda p: [json.loads(l) for l in Path(p).read_text(encoding="utf-8").splitlines() if l.strip()]

JEV_URL = os.getenv("JEV_URL", "https://api.typesafe.ai")
http = requests.Session()
http.mount(JEV_URL, HTTPAdapter(max_retries=Retry(total=4, backoff_factor=0.5, status_forcelist=[408, 429, 500, 502, 503, 504])))

def ask(state, questions, model="jev-latest"):
    r = http.post(f"{JEV_URL}/v1/systemone", timeout=60, headers={"Authorization": f"Bearer {os.environ['TYPESAFE_API_KEY']}"},
                  json={"model": model, "state": state, "questions": questions})
    r.raise_for_status()
    return r.json() | {"_latency_s": r.elapsed.total_seconds()}

choice = lambda instructions, criteria: {"type": "choice", "instructions": instructions, "criteria": criteria}
score = lambda instructions, levels: {"type": "score", "instructions": instructions, "criteria": levels}
noul = lambda instructions, true="", false="": {"type": "noul", "instructions": instructions} | ({"criteria": {"true": true, "false": false}} if true else {})

def probs_of(answer):
    return answer.get("probabilities") or {"true": answer["noul"], "false": 1 - answer["noul"]}

def judge(item, resp):
    (task, q), = item["questions"].items()
    a = resp["answers"][task]
    pr = {str(k): float(v) for k, v in probs_of(a).items()}
    best = max(pr, key=pr.get)
    pred = {"noul": lambda: pr["true"] >= 0.5, "choice": lambda: best, "score": lambda: int(best)}[q["type"]]()
    return {"id": item["id"], "task": item["task"], "qtype": q["type"], "gold": item["gold"], "model": resp["model"],
            "pred": pred, "probs": pr, "conf": pr[best], "conf_field": a.get("confidence"), "correct": bool(pred == item["gold"]),
            "input_tokens": resp["usage"]["input_tokens"], "latency_s": resp["_latency_s"]}

def run(items, engine=lambda it: ask(it["state"], it["questions"]), workers=4):
    with ThreadPoolExecutor(workers) as ex:
        return list(ex.map(lambda it: judge(it, engine(it)), items))

arrays = lambda recs: (np.array([r["conf"] for r in recs]), np.array([r["correct"] for r in recs], float))

def ece(conf, ok, bins=15):
    idx = np.clip(np.searchsorted(np.linspace(0, 1, bins + 1), conf, side="left") - 1, 0, bins - 1)
    return float(sum(abs(conf[idx == i].mean() - ok[idx == i].mean()) * (idx == i).sum() for i in np.unique(idx)) / len(conf))

import re, hashlib, torch
from laya.train import TrainConfig, finetune

MODELS = Path("models"); MODELS.mkdir(exist_ok=True)
(DATA / "distill").mkdir(exist_ok=True)
write_jsonl = lambda p, rows: Path(p).write_text("".join(json.dumps(r, ensure_ascii=False, default=str) + "\n" for r in rows), encoding="utf-8")
FULL = torch.cuda.is_available()
QKEYS = {"choice": lambda q: list(q["criteria"]), "score": lambda q: [str(i) for i in range(len(q["criteria"]))], "noul": lambda q: ["false", "true"]}
top = lambda a: max(probs_of(a), key=probs_of(a).get)          # 질문 유형과 관계없이 확률이 가장 큰 선택지
conf_of = lambda a: max(probs_of(a).values())
p_true = lambda a: probs_of(a)["true"]

def answer_all(engine, states, questions, workers=1):
    with ThreadPoolExecutor(workers) as ex:
        return list(ex.map(lambda s: engine(s, questions), states))

def teacher_rows(name, states, questions):
    # Jev 가 매긴 확률 분포를 그대로 교사 분포로 쓴다(13장 형식)
    resp = answer_all(ask, states, questions, workers=4)
    dist = lambda q, a: {"probabilities": {k: round(float(probs_of(a)[k]), 6) for k in QKEYS[q["type"]](q)}}
    rows = [{"id": f"{name}-{i:05d}", "state": s, "questions": questions, "gold": {n: dist(q, r["answers"][n]) for n, q in questions.items()}}
            for i, (s, r) in enumerate(zip(states, resp))]
    return rows, sum(r["usage"]["input_tokens"] for r in resp)

def distill(name, rows):
    # 학습 파일을 쓰고 multilingual 에서 파인튜닝한다. GPU 가 없으면 200행·1 epoch 의 빠른 모드로 돈다
    path = DATA / "distill" / f"{name}.jsonl"
    write_jsonl(path, random.Random(0).sample(rows, len(rows))[: [200, None][FULL]])
    t0 = time.perf_counter()
    rep = finetune(str(path), "multilingual", str(MODELS / name), TrainConfig(loss="rlcd", epochs=[1, 4][FULL], seed=0))
    return {"학습 항목(질문 단위)": rep["train_items"], "보정 항목": rep["calibration_items"], "epoch 손실": [round(x, 4) for x in rep["epoch_loss"]],
            "시간(초)": round(time.perf_counter() - t0, 1), "데이터 sha256": hashlib.sha256(path.read_bytes()).hexdigest()[:16]}
print("장치:", torch.cuda.get_device_name(0) if FULL else "CPU", "| 모드:", ["빠른 모드", "전체"][FULL])

from typing import TypedDict
from langgraph.graph import StateGraph, START, END

tasks, scenarios = read_jsonl(DATA / "recipes/tasks_ko.jsonl"), read_jsonl(DATA / "recipes/scenarios_ko.jsonl")
AGENTS = {"researcher": "자료·문헌·규정·시장 정보를 찾아 출처와 함께 정리한다",
          "coder": "코드를 고치거나 새로 짜고 테스트한다",
          "writer": "공지·보고서·블로그·메일 같은 글을 쓰고 다듬는다",
          "analyst": "데이터를 집계·분석해 숫자로 원인과 효과를 밝힌다"}
print("단일 작업", len(tasks), "| 시나리오", len(scenarios))

장치: NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition | 모드: 전체
단일 작업 40 | 시나리오 8


## 20.1 위임 질문 설계

작업 하나에 두 가지를 묻는다. 누가 맡을지(choice 4지선다)와 얼마나 급한지(score 5단계, 0 미뤄도 됨 ~ 4 즉시).

In [2]:
AQ = {"assignee": choice("이 작업을 맡을 전문 에이전트", AGENTS),
      "priority": score("이 작업의 긴급도", ["미뤄도 된다", "이번 주 안에", "오늘 안에", "몇 시간 안에", "즉시 (서비스 장애·보안)"])}

def assign_frame(rs):
    return pd.DataFrame([{"작업": t["text"], "정답": t["assignee"], "배정": top(r["answers"]["assignee"]), "확신": conf_of(r["answers"]["assignee"]),
                          "정답 우선순위": t["priority"], "우선순위": int(top(r["answers"]["priority"]))} for t, r in zip(tasks, rs)])

A = assign_frame(answer_all(ask, [t["text"] for t in tasks], AQ, workers=4))
print("배정 정확도:", (A["배정"] == A["정답"]).mean(), "| 우선순위 MAE:", round((A["우선순위"] - A["정답 우선순위"]).abs().mean(), 3),
      "| 한 단계 안:", ((A["우선순위"] - A["정답 우선순위"]).abs() <= 1).mean())
pd.crosstab(A["정답"], A["배정"], margins=True)

배정 정확도: 1.0 | 우선순위 MAE: 0.95 | 한 단계 안: 0.75


배정,analyst,coder,researcher,writer,All
정답,,,,,
analyst,10,0,0,0,10
coder,0,10,0,0,10
researcher,0,0,10,0,10
writer,0,0,0,10,10
All,10,10,10,10,40


## 20.2 LangGraph 슈퍼바이저 그래프

감독 노드는 목표와 지금까지의 기록을 state 로 받아 두 가지를 묻는다. 다음에 누가 일할지(choice)와 목표를 이미 이뤘는지(noul, 7.3절의 done). 조건부 간선이 그 답으로 다음 노드를 고른다. done 이 0.8 을 넘거나 홉이 `max_hops` 를 넘으면 끝난다.

에이전트 노드는 시나리오의 다음 단계가 자기 차례면 정해 둔 결과를 기록에 붙이고, 아니면 「할 일이 없다」를 붙인다. 엔진은 `engine(state, questions)` 모양의 함수면 무엇이든 꽂힌다.

In [3]:
SQ = {"next_agent": choice("목표를 이루기 위해 다음에 일할 에이전트", AGENTS),
      "done": noul("`log` 의 결과로 `goal` 이 이미 모두 이루어졌다", true="목표가 요구한 모든 결과물이 기록에 나타나 있다", false="아직 남은 일이 있다")}

class State(TypedDict):
    goal: str
    log: list
    hops: int
    next: str
    done: float

sup_state = lambda goal, log: {"goal": goal, "log": [f"{x['agent']}: {x['result']}" for x in log]}

def build_graph(engine, scenario, max_hops=6, use_done=True):
    steps = scenario["steps"]

    def supervisor(s):
        a = engine(sup_state(s["goal"], s["log"]), SQ)["answers"]
        return {"next": top(a["next_agent"]), "done": p_true(a["done"]) * use_done, "hops": s["hops"] + 1}

    def worker(name):
        def run_(s):
            k = sum(x["ok"] for x in s["log"])                    # 끝낸 단계 수
            ok = k < len(steps) and steps[k][0] == name
            return {"log": s["log"] + [{"agent": name, "ok": ok, "result": steps[k][1] if ok else "이 단계에서 할 일이 없다"}]}
        return run_

    g = StateGraph(State)
    g.add_node("supervisor", supervisor)
    for name in AGENTS:
        g.add_node(name, worker(name)); g.add_edge(name, "supervisor")
    g.add_edge(START, "supervisor")
    g.add_conditional_edges("supervisor", lambda s: [s["next"], END][s["done"] >= 0.8 or s["hops"] > max_hops], {**{a: a for a in AGENTS}, END: END})
    return g.compile()

def run_scenario(engine, sc, **kw):
    out = build_graph(engine, sc, **kw).invoke({"goal": sc["goal"], "log": [], "hops": 0, "next": "", "done": 0.0}, {"recursion_limit": 50})
    finished, stopped = sum(x["ok"] for x in out["log"]) == len(sc["steps"]), out["done"] >= 0.8
    return {"목표": sc["goal"][:24], "단계 수": len(sc["steps"]), "홉": out["hops"], "헛걸음": sum(not x["ok"] for x in out["log"]), "완료": finished,
            "종료": {(True, True): "정상", (False, True): "조기"}.get((finished, stopped), "홉 초과"),
            "경로": " → ".join(x["agent"] + ["×", ""][x["ok"]] for x in out["log"])}

S20 = pd.DataFrame([run_scenario(ask, sc) for sc in scenarios])
print("완료율:", S20["완료"].mean(), "| 정상 종료:", (S20["종료"] == "정상").mean(), "| 평균 헛걸음:", S20["헛걸음"].mean())
S20

완료율: 0.875 | 정상 종료: 0.375 | 평균 헛걸음: 2.625


,목표,단계 수,홉,헛걸음,완료,종료,경로
0,결제 중복 버그를 고치고 고객 공지를 낸다,2,3,0,True,정상,coder → writer
1,지난 분기 이탈 원인을 분석하고 경영진 보고,2,7,4,True,홉 초과,analyst → writer → writer× → writer× → writer×...
2,경쟁사 가격을 조사하고 우리 요금제의 수익성,2,7,4,True,홉 초과,researcher → analyst → analyst× → analyst× → a...
3,CSV 한글 깨짐을 고치고 FAQ 를 갱신한,2,3,0,True,정상,coder → writer
4,새 개인정보 규제를 확인하고 영향받는 코드를,2,7,4,True,홉 초과,researcher → coder → coder× → coder× → coder× ...
5,서버 비용 증가 원인을 찾아 절감안을 쓴다,3,7,5,False,홉 초과,analyst → writer× → writer× → writer× → writer...
6,논문 다섯 편을 조사해 기술 블로그 글을 쓴,2,7,4,True,홉 초과,researcher → writer → writer× → writer× → writ...
7,앱 크래시를 고친다,1,2,0,True,정상,coder


## 20.3 루프 종료 조건

같은 그래프에서 done 질문의 답을 무시하고(`use_done=False`) 홉 상한만으로 끝내 본다. 일이 다 끝난 뒤에도 감독이 누군가에게 일을 계속 맡기며 헛걸음을 쌓는다.

In [4]:
no_done = pd.DataFrame([run_scenario(ask, sc, use_done=False) for sc in scenarios])
pd.DataFrame({"done 판단 있음": {"평균 홉": S20["홉"].mean(), "평균 헛걸음": S20["헛걸음"].mean(), "완료율": S20["완료"].mean()},
              "홉 상한만": {"평균 홉": no_done["홉"].mean(), "평균 헛걸음": no_done["헛걸음"].mean(), "완료율": no_done["완료"].mean()}}).T

,평균 홉,평균 헛걸음,완료율
done 판단 있음,5.375,2.625,0.875
홉 상한만,7.000,4.250,0.875


## 20.4 용도별 증류

감독이 보는 입력은 두 가지다. 작업 한 문장(20.1)과, 목표에 실행 기록이 붙은 상태(20.2). 둘 다 로컬 LLM 으로 만든다.

1. **작업 문장.** 에이전트 4 × 업종 10 × 긴급함 2 = 프롬프트 80개, 프롬프트마다 10줄이다.
2. **결과 보고.** 작업 문장마다 「이 일을 끝낸 담당자가 남길 한 줄 결과」를 생성한다.
3. **실행 기록.** 서로 다른 에이전트의 작업 둘을 이어 목표 하나로 만든다. 기록이 빈 상태, 첫 결과만 붙은 상태, 둘 다 붙은 상태 셋을 만든다. 마지막 상태가 done 을 가르친다.

다음에 일할 에이전트와 done 의 라벨은 Jev 가 매긴다. 생성 의도(어느 에이전트의 일로 시킨 문장인지, 결과가 몇 개 붙었는지)는 점검용으로만 본다. 평가용 작업·시나리오에 대한 로컬 LLM 의 성적도 모델이 올라와 있는 동안 잰다.

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer

LLM_MODEL = os.environ["LOCAL_LLM_MODEL"]
tok = AutoTokenizer.from_pretrained(LLM_MODEL, padding_side="left")
tok.pad_token = tok.pad_token or tok.eos_token
lm = AutoModelForCausalLM.from_pretrained(LLM_MODEL, dtype="auto", device_map="auto")

def chat_batch(prompts, max_new_tokens=256, batch_size=16, system=None):
    # 프롬프트 여럿을 묶어 generate 한 번에 보낸다(왼쪽 패딩). 서빙 엔진 없이 처리량을 내는 방법이다
    msgs = lambda p: [{"role": "system", "content": system}] * bool(system) + [{"role": "user", "content": p}]
    texts = [tok.apply_chat_template(msgs(p), tokenize=False, add_generation_prompt=True, enable_thinking=False) for p in prompts]
    outs = []
    for i in range(0, len(texts), batch_size):
        enc = tok(texts[i:i + batch_size], return_tensors="pt", padding=True).to(lm.device)
        gen = lm.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False)
        outs += tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return [o.strip() for o in outs]

chat = lambda prompt, **kw: chat_batch([prompt], **kw)[0]
print(LLM_MODEL, "|", lm.device)

import re

KEYS = {"choice": lambda q: list(q["criteria"]), "score": lambda q: [str(i) for i in range(len(q["criteria"]))], "noul": lambda q: ["true", "false"]}
LINES = {"choice": lambda q: [f"  {k}: {v}" for k, v in q["criteria"].items()],
         "score": lambda q: [f"  {i}: {v}" for i, v in enumerate(q["criteria"])],
         "noul": lambda q: [f"  {k}: {v}" for k, v in (q.get("criteria") or {"true": "그렇다", "false": "아니다"}).items()]}

def gen_prompt(state, q):
    return "\n".join(["[판단할 대상]", json.dumps(state, ensure_ascii=False), "", "[질문]", q["instructions"], "[선택지]", *LINES[q["type"]](q), "",
                       "선택지 키 하나와, 그 답이 맞을 확률을 아래 두 줄 형식으로만 답하라.", "답: <선택지 키>", "확신: <0과 1 사이 숫자>"])

def gen_answer(q, text):
    keys = KEYS[q["type"]](q)
    ans = (re.findall(r"답\s*[:：]\s*(\S+)", text) + [None])[0]
    conf = float((re.findall(r"확신\s*[:：]\s*([01](?:\.\d+)?)", text) + ["nan"])[0])
    p = {k: [(1 - conf) / (len(keys) - 1), conf][k == ans] for k in keys} if ans in keys and conf == conf else {k: 1 / len(keys) for k in keys}
    return {"type": q["type"], "probabilities": p, "noul": p.get("true"), "choice": max(p, key=p.get), "invalid": ans not in keys, "raw": text}

def gen_many(states_questions, batch_size=16):
    # (state, 질문 하나짜리 dict) 목록을 묶어서 생성하고 Jev 모양의 응답으로 바꾼다
    flat = [(s, n, q) for s, qs in states_questions for n, q in qs.items()]
    t0 = time.perf_counter()
    outs = chat_batch([gen_prompt(s, q) for s, _, q in flat], max_new_tokens=24, batch_size=batch_size)
    sec = (time.perf_counter() - t0) / len(flat)
    return [{"model": "llm:" + LLM_MODEL, "answers": {n: gen_answer(q, o)}, "usage": {"input_tokens": 0}, "_latency_s": sec}
            for (s, n, q), o in zip(flat, outs)]

def llm_answers(states, questions, batch_size=16):
    flat, n = gen_many([(s, questions) for s in states], batch_size), len(questions)
    return [{"model": flat[0]["model"], "answers": {k: v for r in flat[i:i + n] for k, v in r["answers"].items()},
             "usage": {"input_tokens": 0}, "_latency_s": sum(r["_latency_s"] for r in flat[i:i + n])} for i in range(0, len(flat), n)]

def lines_of(text):
    # 생성한 목록에서 번호·기호를 떼고 한 줄에 하나씩 꺼낸다
    return [l for l in (re.sub(r"^\s*(\d+[.)]|[-*•])\s*", "", x).strip().strip('"') for x in text.splitlines()) if 4 <= len(l) <= 200]

generate = lambda prompts, max_new_tokens=512: chat_batch(prompts, max_new_tokens=max_new_tokens, batch_size=16)

SECTORS = ["전자상거래", "핀테크", "게임", "병원", "제조", "물류", "교육", "미디어", "공공기관", "SaaS 스타트업"]
plan = [(a, f"{s} 회사의 업무 요청 가운데 '{desc}' 담당에게 맡길 {u} 작업 10개를 써라. 한 줄에 하나씩, 번호 없이 작업 문장만 쓴다.")
        for a, desc in AGENTS.items() for s in SECTORS for u in ["급한", "급하지 않은"]]
t0 = time.perf_counter()
made = [(a, l) for (a, _), text in zip(plan, generate([p for _, p in plan])) for l in lines_of(text)]
task_text = list(dict.fromkeys(l for _, l in made))
intent = {l: a for a, l in made}
results = {t: (lines_of(r) + ["완료"])[0] for t, r in zip(task_text, generate([f"다음 작업을 끝낸 담당자가 남길 한 줄 결과 보고를 써라. 수치나 산출물을 구체적으로 쓴다.\n작업: {t}" for t in task_text], 80))}
print(f"작업 {len(task_text)}개 ({time.perf_counter() - t0:.0f}초)")
pd.DataFrame([{"의도": intent[t], "작업": t, "결과": results[t]} for t in task_text]).groupby("의도").head(2)

/home/josh/anaconda3/envs/pytorch/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Ignoring corrupted tree cache file /data/shared/hf/hub/models--Qwen--Qwen3.5-4B/trees/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a.json: [Errno 13] Permission denied: '/data/shared/hf/hub/models--Qwen--Qwen3.5-4B/trees/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a.json'
Loading weights: 100%|██████████| 426/426 [00:01<00:00, 377.58it/s]


Qwen/Qwen3.5-4B | cuda:0


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


작업 752개 (171초)


,의도,작업,결과
0,researcher,신제품 출시를 위한 경쟁사 가격 비교 자료와 시장 점유율 통계 정보를 수집하여 출처...,"경쟁사 5 개사 평균 가격 대비 12% 낮게 책정된 신제품 가격 전략과, 시장 점유..."
1,researcher,고객 문의에 대한 답변을 위한 최신 제품 사용 가이드와 FAQ 문서를 정리하고 관련...,"최신 제품 사용 가이드 2 장과 FAQ 15 문항을 정리하며, 관련 규정 3 개를 ..."
189,coder,사용자 로그인 시 비밀번호 입력 오류가 발생하여 즉시 수정하고 테스트해야 합니다.,비밀번호 입력 오류 수정 완료 및 정상 로그인 테스트 통과 (100% 성공률).
190,coder,결제 페이지에서 카드 번호가 16 자 이상일 때 오류가 뜨는 버그를 고쳐야 합니다.,카드 번호 16 자 이상 입력 시 발생하는 오류를 수정하여 정상 처리 완료.
375,writer,새로운 제품 출시 기념으로 고객들에게 보내는 긴급 공지문 초안 작성,"새 제품 출시 기념 긴급 공지문 초안을 작성 완료했으며, 발송 대상 1,200 명에..."
376,writer,이번 주 금요일까지 마감되는 월간 실적 보고서의 핵심 데이터 정리 및 요약 작성,"이번 주 금요일까지 월간 실적 보고서 핵심 데이터 12 개 항목을 정리하고, 총 매..."
562,analyst,지난 3 개월 판매 데이터로 각 카테고리별 매출 추이와 성장 요인을 분석한다.,"지난 3 개월 매출은 총 12 억 4,500 만 원으로 전년 동기 대비 8.2% 성..."
563,analyst,고객 이탈 원인을 규명하기 위해 최근 6 개월의 문의 로그를 집계한다.,"최근 6 개월 문의 로그를 집계하여 총 1,240 건을 분석하고, 이탈 원인은 '결..."


In [6]:
rng = random.Random(2026)
pairs = [(t1, t2) for t1, t2 in (rng.sample(task_text, 2) for _ in range(300)) if intent[t1] != intent[t2]]
log_of = lambda ts: [{"agent": intent[t], "result": results[t]} for t in ts]
sup_states = [(sup_state(f"{t1}. 이어서 {t2}", log_of([t1, t2][:k])), k) for t1, t2 in pairs for k in range(3)]
print("감독 상태", len(sup_states))
sup_states[1][0]

감독 상태 678


{'goal': '유아 교육 관련 최신 연구 논문과 정책 변경 사항을 정리한 보고서 작성. 이어서 서버 로그에서 중요한 오류 메시지를 필터링하여 모니터링 대시보드에 실시간으로 전송되도록 구현한다.',
 'log': ['researcher: 최신 유아 교육 연구 논문 12 건 및 정책 변경 사항 5 건을 분석하여 종합 보고서 1 장을 작성 완료했습니다.']}

In [7]:
llm_engine = lambda state, questions: llm_answers([state], questions)[0]
llm_row = {"assign": llm_answers([t["text"] for t in tasks], AQ), "sc": pd.DataFrame([run_scenario(llm_engine, sc) for sc in scenarios]),
           "latency": [r["_latency_s"] for r in llm_answers([sup_state(s["goal"], []) for s in scenarios], SQ)]}
del lm; torch.cuda.empty_cache()

In [8]:
task_rows, t_tokens = teacher_rows("task", task_text, AQ)
sup_rows, s_tokens = teacher_rows("supervisor", [s for s, _ in sup_states], SQ)
check = {"작업: Jev 배정 = 생성 의도": np.mean([max(g := r["gold"]["assignee"]["probabilities"], key=g.get) == intent[r["state"]] for r in task_rows]),
         "감독: 결과 둘일 때 done ≥ 0.5": np.mean([r["gold"]["done"]["probabilities"]["true"] >= 0.5 for r, (_, k) in zip(sup_rows, sup_states) if k == 2]),
         "감독: 결과 하나 이하일 때 done < 0.5": np.mean([r["gold"]["done"]["probabilities"]["true"] < 0.5 for r, (_, k) in zip(sup_rows, sup_states) if k < 2])}
print(f"학습 행 {len(task_rows) + len(sup_rows)} | Jev 비용 {(t_tokens + s_tokens) * 0.042 / 1e6:.4f}달러", {k: round(float(v), 3) for k, v in check.items()})
pd.Series(distill("delegation-distill", task_rows + sup_rows))

학습 행 1430 | Jev 비용 0.0336달러 {'작업: Jev 배정 = 생성 의도': 0.834, '감독: 결과 둘일 때 done ≥ 0.5': 0.898, '감독: 결과 하나 이하일 때 done < 0.5': 1.0}


Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 2364.05it/s]


train items 2574, calibration items 286, eval items 286 (calibration_slice), skipped {}, device cuda:0
optimizer budget: 2574 train item(s), effective batch up to 64 (micro_batch 8 x grad_accum 8), 164 optimizer updates over 4 epoch(s)


/home/josh/anaconda3/envs/pytorch/lib/python3.11/site-packages/laya/train.py:724: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


epoch 1/4 step 100 loss 0.5836
epoch 1/4 step 200 loss 0.0459
epoch 1/4 step 300 loss 0.2518
epoch 1/4 mean loss 0.7619
epoch 2/4 step 100 loss 0.2782
epoch 2/4 step 200 loss 0.0969
epoch 2/4 step 300 loss 0.0010
epoch 2/4 mean loss 0.4562
epoch 3/4 step 100 loss 0.1241
epoch 3/4 step 200 loss 0.4572
epoch 3/4 step 300 loss 0.5294
epoch 3/4 mean loss 0.3937
epoch 4/4 step 100 loss 0.5994
epoch 4/4 step 200 loss 0.4242
epoch 4/4 step 300 loss 0.2786
epoch 4/4 mean loss 0.4021

=== Calibration Evidence (in-sample calibration slice, 286 items; not held-out) ===
Metric                 Before         After          Delta         
-----------------------------------------------------------------
Accuracy               0.3916         0.8252         +0.4336       
Loss                   2.4527         0.5639         -1.8888       
ECE                    0.2698         0.0661         -0.2037       
Brier (multi-class)    0.6024         0.0719         -0.5305       
Brier (top-1)          0.3197

학습 항목(질문 단위)                                2574
보정 항목                                        286
epoch 손실        [0.7619, 0.4562, 0.3937, 0.4021]
시간(초)                                      124.4
데이터 sha256                      915702f1997882cc
dtype: object

## 20.5 엔진 비교

감독 자리의 엔진만 바꾼다. 단일 작업의 배정 정확도, 여러 단계 작업의 완료율과 정상 종료율, 감독 판단 한 번의 지연을 본다. 로컬 LLM 줄은 20.4 에서 잰 값이다.

In [9]:
import laya
from laya import Router

router = Router()                          # 한국어는 multilingual 체크포인트로 간다(10장)

def laya_post(state, questions):
    t0 = time.perf_counter()
    r = router.predict(state, questions)
    return r | {"model": "laya:" + r["routing"]["model"], "usage": {"input_tokens": 0}, "_latency_s": time.perf_counter() - t0}
print("laya", laya.__version__)

import laya

def checkpoint_engine(path, **load_kwargs):
    agent = laya.load(str(path), **load_kwargs)
    def post(state, questions):
        t0 = time.perf_counter()
        r = agent.predict(state, questions)
        return r | {"model": "laya:" + Path(path).name, "usage": {"input_tokens": 0}, "_latency_s": time.perf_counter() - t0}
    post.agent = agent
    return post

def delegation_report(assign, sc, latency):
    A_ = assign_frame(assign)
    return {"배정 정확도": (A_["배정"] == A_["정답"]).mean(), "우선순위 MAE": (A_["우선순위"] - A_["정답 우선순위"]).abs().mean(),
            "시나리오 완료율": sc["완료"].mean(), "정상 종료율": (sc["종료"] == "정상").mean(), "평균 헛걸음": sc["헛걸음"].mean(),
            "감독 지연 중앙값(ms)": 1000 * np.median(latency)}

def measure(engine, workers=1):
    return delegation_report(answer_all(engine, [t["text"] for t in tasks], AQ, workers), pd.DataFrame([run_scenario(engine, sc) for sc in scenarios]),
                             [r["_latency_s"] for r in answer_all(engine, [sup_state(s["goal"], []) for s in scenarios], SQ)])

ENGINES = {"Jev": ask, "Laya 기본": laya_post, "Laya 증류": checkpoint_engine(MODELS / "delegation-distill")}
pd.DataFrame({n: measure(e, {"Jev": 4}.get(n, 1)) for n, e in ENGINES.items()} | {"로컬 LLM": delegation_report(**llm_row)}).T

laya 0.4.7


Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 1824.09it/s]


,배정 정확도,우선순위 MAE,시나리오 완료율,정상 종료율,평균 헛걸음,감독 지연 중앙값(ms)
Jev,1.000,0.950,0.875,0.375,2.625,159.069
Laya 기본,0.575,1.200,0.000,0.000,3.125,14.674
Laya 증류,0.900,1.275,0.125,0.000,5.375,14.208
로컬 LLM,1.000,0.800,0.000,0.000,0.000,97.744


## 정리

이 노트북이 잰 것과 만든 것이다. 결과 해석은 20장 본문에서 한다.

- 20.1 단일 작업 배정 정확도·혼동표, 우선순위 MAE
- 20.2 시나리오별 홉·헛걸음·완료·종료 방식과 경로
- 20.3 done 판단이 있을 때와 홉 상한만 있을 때의 홉·헛걸음
- 20.4 로컬 LLM 이 만든 작업·결과·감독 상태, 생성 의도와 Jev 라벨의 일치, 증류 체크포인트(`models/delegation-distill`)
- 20.5 네 엔진의 배정 정확도·완료율·정상 종료율·감독 지연